In [1]:
import sys
sys.path.append('../tracking/tracking_utils')

from utils import relabel_sequential_lineage, get_image_features

from scipy.spatial.distance import cdist

import os

import io
import json
import tarfile

import numpy as np

from numcodecs import JSON
import zarr

from skimage.segmentation import relabel_sequential

from tqdm import tqdm
import glob

def get_features(X, y, lineages, max_cells,  appearance_shape=(16,16,1), crop_mode='fixed', distance_threshold=64):
        """
        Extract the relevant features from the label movie
        Appearance, morphologies, centroids, and adjacency matrices
        """
        max_tracks = max_cells
        n_batches = X.shape[0]
        n_frames = X.shape[1]
        n_channels = X.shape[-1]

        batch_shape = (n_batches, n_frames, max_tracks)

        appearances = np.zeros(batch_shape + appearance_shape, dtype='float32')

        morphologies = np.zeros(batch_shape + (3,), dtype='float32')

        centroids = np.zeros(batch_shape + (2,), dtype='float32')

        adj_matrix = np.zeros(batch_shape + (max_tracks,), dtype='float32')

        temporal_adj_matrix = np.zeros((n_batches,
                                        n_frames - 1,
                                        max_tracks,
                                        max_tracks,
                                        3), dtype='float32')

        mask = np.zeros(batch_shape + (max_tracks,), dtype='bool')

        track_length = np.zeros((n_batches, max_tracks, 2), dtype='int32')

        for batch in tqdm(range(n_batches)):
            for frame in range(n_frames):

                frame_features = get_image_features(
                    X[batch, frame], y[batch, frame],
                    appearance_dim=appearance_shape[1],
                    crop_mode=crop_mode)

                track_ids = frame_features['labels'] - 1
                centroids[batch, frame, track_ids] = frame_features['centroids']
                morphologies[batch, frame, track_ids] = frame_features['morphologies']
                appearances[batch, frame, track_ids] = frame_features['appearances']
                if track_ids.size != 0:
                    mask[batch, frame, :track_ids[-1], :track_ids[-1]] = 1

                # Get adjacency matrix, cannot filter on track ids.
                cent = centroids[batch, frame]
                distance = cdist(cent, cent, metric='euclidean')
                distance = distance < distance_threshold

                # Disconnect the padded nodes
                morphs = morphologies[batch, frame]
                is_pad = np.matmul(morphs, morphs.T) == 0

                adj = distance * (1 - is_pad)
                adj_matrix[batch, frame] = adj.astype(np.float32)

            # Get track length and temporal adjacency matrix
            for label in lineages[batch]:

                # Get track length
                start_frame = lineages[batch][label]['frames'][0]
                end_frame = lineages[batch][label]['frames'][-1]

                track_id = int(label) - 1
                track_length[batch, track_id, 0] = start_frame
                track_length[batch, track_id, 1] = end_frame

                # Get temporal adjacency matrix
                frames = lineages[batch][label]['frames']

                # Assign same
                for f0, f1 in zip(frames[0:-1], frames[1:]):
                    if f1 - f0 == 1:
                        temporal_adj_matrix[batch, f0, track_id, track_id, 0] = 1

                # Assign daughter
                # WARNING: This wont work if there's a time gap between mother
                # cell disappearing and daughter cells appearing
                last_frame = frames[-1]
                daughters = lineages[batch][label]['daughters']
                for daughter in daughters:
                    daughter_id = daughter - 1
                    temporal_adj_matrix[batch, last_frame, track_id, daughter_id, 2] = 1

            # Assign different
            same_prob = temporal_adj_matrix[batch, ..., 0]
            daughter_prob = temporal_adj_matrix[batch, ..., 2]
            temporal_adj_matrix[batch, ..., 1] = 1 - same_prob - daughter_prob

            # Identify cell padding
            for i in range(temporal_adj_matrix.shape[2]):
                # index + 1 is the cell label
                if i + 1 not in lineages[batch]:
                    temporal_adj_matrix[batch, :, i] = 0
                    temporal_adj_matrix[batch, :, :, i] = 0

            # Identify temporal padding
            for b in range(temporal_adj_matrix.shape[0]):
                sames = temporal_adj_matrix[b, ..., 0]
                sames = np.sum(sames, axis=(1, 2))
                temporal_adj_matrix[b, sames == 0] = 0

        features = {
            'adj_matrix': adj_matrix, # adjacency matrix for graph
            'appearances': appearances, # crops for cells
            'morphologies': morphologies, # corresponding morphologies
            'centroids': centroids, # corresponding centroids
            'labels': temporal_adj_matrix, # label matrix
            'mask': mask, # mask for including data in loss function
            'track_length': track_length # track lengths for generating samples
            }

        return features


def load_trks(filename):
    """Load a trk/trks file.

    Args:
        filename (str or BytesIO): full path to the file including .trk/.trks
            or BytesIO object with trk file data

    Returns:
        dict: A dictionary with raw, tracked, and lineage data.
    """

    if isinstance(filename, io.BytesIO):
        kwargs = {'fileobj': filename}
    else:
        kwargs = {'name': filename}

    with tarfile.open(mode='r', **kwargs) as trks:

        # numpy can't read these from disk...
        with io.BytesIO() as array_file:
            array_file.write(trks.extractfile('raw.npy').read())
            array_file.seek(0)
            raw = np.load(array_file)

        with io.BytesIO() as array_file:
            array_file.write(trks.extractfile('tracked.npy').read())
            array_file.seek(0)
            tracked = np.load(array_file)

        # trks.extractfile opens a file in bytes mode, json can't use bytes.
        try:
            trk_data = trks.getmember('lineages.json')
        except KeyError:
            try:
                trk_data = trks.getmember('lineage.json')
            except KeyError:
                raise ValueError('Invalid .trk file, no lineage data found.')

        lineages = json.loads(trks.extractfile(trk_data).read().decode())
        lineages = lineages if isinstance(lineages, list) else [lineages]

        # JSON only allows strings as keys, so convert them back to ints
        # for i, tracks in enumerate(lineages):
        #     lineages[i] = {str(k): v for k, v in tracks.items()}

    return {'lineages': lineages, 'X': raw, 'y': tracked}

def correct_lineages(X, y, lineages):
    """Ensure valid lineages and sequential labels for all batches"""
    new_X = []
    new_y = []
    new_lineages = []
    for batch in tqdm(range(y.shape[0])):
        # if is_valid_lineage(self.y[batch], self.lineages[batch]):

        y_relabel, new_lineage = relabel_sequential_lineage(
            y[batch], lineages[batch])

        new_X.append(X[batch])
        new_y.append(y_relabel)
        new_lineages.append(new_lineage)
        # else:
        #     print('Invalid lineage detected.')

    X = np.stack(new_X, axis=0)
    y = np.stack(new_y, axis=0)
    lineages = new_lineages

    return X, y, lineages

def convert_trk_to_zarr(filename, out_dir=None):

    if out_dir is None:
        file_dir = os.path.dirname(filename)
        split = os.path.splitext(os.path.basename(filename))[0]
        output_file = os.path.join(file_dir, split) + '_proc.zarr'
        output_json = os.path.join(file_dir, split) + '.json'

    data = load_trks(filename)

    X = data['X']
    y = data['y']
    lineages = data['lineages']
    
    # Step 1: correct lineages function
    X, y, lineages = correct_lineages(X, y, lineages)

    # Step 2: get maximum cells from lineages
    max_cells = 0
    for i in lineages:
        curr_m = len(i.keys())
        if curr_m > max_cells:
            max_cells = curr_m

    features = get_features(X, y, lineages, max_cells)

    z = zarr.open(output_file)

    for k, v in features.items():
        
        print(f"{k}: {v.shape}")

        z.create_array(k, data=v)

for filename in glob.glob('../data/DynamicNuclearNet-tracking-v1_0/*.trks'):
    print(f"Converting {os.path.basename(filename)}")
    convert_trk_to_zarr(filename)

Converting test.trks


100%|██████████| 12/12 [00:16<00:00,  1.39s/it]


adj_matrix: (12, 71, 331, 331)
appearances: (12, 71, 331, 16, 16, 1)
morphologies: (12, 71, 331, 3)
centroids: (12, 71, 331, 2)
labels: (12, 70, 331, 331, 3)
mask: (12, 71, 331, 331)
track_length: (12, 331, 2)
Converting train.trks


 85%|████████▍ | 77/91 [02:48<00:42,  3.06s/it]/data/sholtzen/torch-tracking/.venv/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/data/sholtzen/torch-tracking/.venv/lib/python3.11/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/data/sholtzen/torch-tracking/.venv/lib/python3.11/site-packages/numpy/_core/_methods.py:223: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/data/sholtzen/torch-tracking/.venv/lib/python3.11/site-packages/numpy/_core/_methods.py:181: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/data/sholtzen/torch-tracking/.venv/lib/python3.11/site-packages/numpy/_core/_methods.py:215: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcoun

adj_matrix: (91, 71, 405, 405)
appearances: (91, 71, 405, 16, 16, 1)
morphologies: (91, 71, 405, 3)
centroids: (91, 71, 405, 2)
labels: (91, 70, 405, 405, 3)
mask: (91, 71, 405, 405)
track_length: (91, 405, 2)
Converting val.trks


100%|██████████| 27/27 [00:34<00:00,  1.29s/it]


adj_matrix: (27, 71, 277, 277)
appearances: (27, 71, 277, 16, 16, 1)
morphologies: (27, 71, 277, 3)
centroids: (27, 71, 277, 2)
labels: (27, 70, 277, 277, 3)
mask: (27, 71, 277, 277)
track_length: (27, 277, 2)


In [ ]:
import sys

sys.path.append('../tracking/tracking_utils/')
sys.path.append('../data')

from loader import create_trk_dataloaders


config = {
    'optimizer': 'radam',
    'learning_rate': 1e-3,
    'weight_decay': 0,
    'decay': 0.99,
    'scheduler': 'reduce_on_plateau',
    'max_epochs': 50,
    'batch_size': 8,
    'n_layers': 1,
    'num_workers': 16,
    'clipnorm': 1e-3,
    'step_size': 5,
    'crop_mode': 'fixed',
    'patience': 5,
    'log_and_save': True,
    'enable_early_stopping': True,
    'crop_size': 16,
    'attention': False,
    'truncate_dataset': None,
    'loss': 'wcce',
    't_direction': 'forward',
    'processed': True
}


_, val_loader, _ = create_trk_dataloaders(
    val_path='../data/DynamicNuclearNet-tracking-v1_0/val_proc.zarr',
    batch_size=config['batch_size'],
    distance_threshold=64,
    augment=False,
    crop_mode=config['crop_mode'],
    num_workers=config['num_workers'],
    crop_size=config['crop_size'],
    truncate_dataset = config['truncate_dataset'],
    t_direction=config['t_direction'],
    processed=config['processed']
        )

In [ ]:
dataloader = iter(val_loader)

batch = next(dataloader)

In [ ]:
labels = batch['labels']

In [ ]:
print(labels.shape)

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(labels[1,0,])
print(labels[1,0,].min())
plt.figure()